In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from src import synthetic, features as F

In [2]:
FORECAST_HORIZON = 10
NUM_STORES = 4
NUM_DAYS = 365*4

# Generate sales features

In [3]:

stores = synthetic.store_data(n_stores=NUM_STORES)
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)

stores

,Store,StoreType,Assortment,CompetitionDistance,CompetitionSinceDate,Promo2SinceDate,PromoInterval,Promo2
0,1,a,a,500.0,2012-04-01,2012-07-01,"Jan,Apr,Jul,Oct",1
1,2,b,a,850.0,2012-07-01,NaT,,0
2,3,c,b,1200.0,2012-10-01,2013-07-01,"Jan,Apr,Jul,Oct",1
3,4,a,a,1550.0,2013-01-01,NaT,,0


In [4]:

stores = synthetic.store_data(n_stores=NUM_STORES)
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)
sales_stores = sales.merge(stores, on='Store', how='left').set_index(['Store', 'Date'])
sales_stores = sales_stores.sample(frac=1, random_state=42)  # Shuffle the data

no_state_holiday = sales_stores['StateHoliday'].isin(['0', 0]) | sales_stores['StateHoliday'].isna()
sales_stores.loc[no_state_holiday, 'StateHoliday'] = 'NoHoliday'
sales_stores['isStateHoliday'] = sales_stores['StateHoliday'] != 'NoHoliday'
sales_stores['isSchoolHoliday'] = sales_stores['SchoolHoliday'].astype(bool)

sales_stores['CompetitionStartDate'] = sales_stores['CompetitionSinceDate'].apply(lambda x: pd.to_datetime(x) if pd.notnull(x) else pd.NaT)

sales_stores.loc[sales_stores['Sales'] == 0, 'Sales'] = np.nan
sales_stores.ffill(inplace=True)  # Fill NaN values with the last valid observation


In [8]:
""" Past features """

lags = [pd.DateOffset(days=i) for i in range(1, 3)]
diffs = [pd.DateOffset(days=i) for i in range(1, 3)]
windows = ['7D', '14D', '30D']


one_day_offset = pd.DateOffset(days=1)  # We use this to shift the rolling windows and differences by one day to avoid data leakage from the current day.
forecast_offset = pd.DateOffset(days=FORECAST_HORIZON) # We use this to shift the features by the forecast horizon to align them with the target variable.


grouped = sales_stores.reset_index('Store').groupby('Store')

x_lag = grouped['Sales'].apply(F.lags, lags=lags)
x_dif = grouped['Sales'].apply(F.diffs, diffs=diffs, lag=one_day_offset)

x_window = grouped['Sales'].apply(F.rolling, windows=windows, agg_func='mean', lag=one_day_offset)
x_calendar = grouped['Sales'].apply(lambda x: F.calendar(x.index.to_series()+forecast_offset))

x_competition = grouped['CompetitionStartDate'].apply(F.days_with_competition, offset=forecast_offset)
x_state_hol = grouped['isStateHoliday'].apply(F.holiday_counters, offset=forecast_offset)
x_school_hol = grouped['SchoolHoliday'].apply(F.holiday_counters, offset=forecast_offset)
x_promo = grouped['Promo'].apply(F.days_in_promotion, offset=forecast_offset)
x_promo2 = grouped['Promo2'].apply(F.days_in_promotion, offset=forecast_offset)

x_synth = pd.concat([x_lag, x_dif, x_window, x_calendar, x_competition, x_state_hol, x_school_hol, x_promo, x_promo2], axis=1)

x_synth.sort_index().head(10)

lag_days_1  lag_days_2  diff_days_1  diff_days_2  \
Store Date                                                           
1     2013-01-01         NaN         NaN          NaN          NaN   
      2013-01-02         0.0         NaN          NaN          NaN   
      2013-01-03      5530.0         0.0       5530.0          NaN   
      2013-01-04      4327.0      5530.0      -1203.0       4327.0   
      2013-01-05      4486.0      4327.0        159.0      -1044.0   
      2013-01-06      4997.0      4486.0        511.0        670.0   
      2013-01-07         0.0      4997.0      -4997.0      -4486.0   
      2013-01-08      7176.0         0.0       7176.0       2179.0   
      2013-01-09      5580.0      7176.0      -1596.0       5580.0   
      2013-01-10      5471.0      5580.0       -109.0      -1705.0   

                  rolling_mean_7D  rolling_mean_14D  rolling_mean_30D  year  \
Store Date                                                                    
1     2013-01-01              NaN               NaN               NaN  2013   
      2013-01-02         0.000000          0.000000          0.000000  2013   
      2013-01-03      2765.000000       2765.000000       2765.000000  2013   
      2013-01-04      3285.666667       3285.666667       3285.666667  2013   
      2013-01-05      3585.750000       3585.750000       3585.750000  2013   
      2013-01-06      3868.000000       3868.000000       3868.000000  2013   
      2013-01-07      3223.333333       3223.333333       3223.333333  2013   
      2013-01-08      3788.000000       3788.000000       3788.000000  2013   
      2013-01-09      4585.142857       4012.000000       4012.000000  2013   
      2013-01-10      4576.714286       4174.111111       4174.111111  2013   

                  quarter  month  ...  is_weekend  is_weekday  is_month_end  \
Store Date                        ...                                         
1     2013-01-01        1      1  ...       False        True         False   
      2013-01-02        1      1  ...        True       False         False   
      2013-01-03        1      1  ...        True       False         False   
      2013-01-04        1      1  ...       False        True         False   
      2013-01-05        1      1  ...       False        True         False   
      2013-01-06        1      1  ...       False        True         False   
      2013-01-07        1      1  ...       False        True         False   
      2013-01-08        1      1  ...       False        True         False   
      2013-01-09        1      1  ...        True       False         False   
      2013-01-10        1      1  ...        True       False         False   

                  CompetitionStartDate  DaysToNextHoliday  \
Store Date                                                  
1     2013-01-01                1593.0               77.0   
      2013-01-02                1594.0               76.0   
      2013-01-03                1595.0               75.0   
      2013-01-04                1596.0               74.0   
      2013-01-05                1597.0               73.0   
      2013-01-06                1598.0               72.0   
      2013-01-07                1599.0               71.0   
      2013-01-08                1600.0               70.0   
      2013-01-09                1601.0               69.0   
      2013-01-10                1602.0               68.0   

                  DaysSinceLastHoliday  DaysToNextHoliday  \
Store Date                                                  
1     2013-01-01                  10.0                0.0   
      2013-01-02                  11.0               72.0   
      2013-01-03                  12.0               71.0   
      2013-01-04                  13.0               70.0   
      2013-01-05                  14.0               69.0   
      2013-01-06                  15.0               68.0   
      2013-01-07                  16.0               67.0 

In [6]:
from src import constants as C


sales = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
stores = pd.read_csv(C.STORE_FILE)
sales_stores = sales.merge(stores, on='Store', how='left').set_index(['Store', 'Date'])

no_state_holiday = sales_stores['StateHoliday'].isin(['0', 0]) | sales_stores['StateHoliday'].isna()
sales_stores.loc[no_state_holiday, 'StateHoliday'] = 'NoHoliday'
sales_stores['isStateHoliday'] = sales_stores['StateHoliday'] != 'NoHoliday'
sales_stores['isSchoolHoliday'] = sales_stores['SchoolHoliday'].astype(bool)

sales_stores['CompetitionStartDate'] = pd.to_datetime(
    sales_stores[['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth']]
    .rename(columns={'CompetitionOpenSinceYear': 'year', 'CompetitionOpenSinceMonth': 'month'})
    .assign(day=1)
)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_12576\2969052399.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [7]:
grouped = sales_stores.reset_index('Store').groupby('Store')

x_lag = grouped['Sales'].apply(F.lags, lags=lags)
x_dif = grouped['Sales'].apply(F.diffs, diffs=diffs, lag=one_day_offset)
x_window = grouped['Sales'].apply(F.rolling, windows=windows, agg_func='mean', lag=one_day_offset)
x_calendar = grouped['Sales'].apply(lambda x: F.calendar(x.index.to_series()+forecast_offset))

x_competition = grouped['CompetitionStartDate'].apply(F.days_with_competition, offset=forecast_offset)
x_state_hol = grouped['isStateHoliday'].apply(F.holiday_counters, offset=forecast_offset)
x_school_hol = grouped['SchoolHoliday'].apply(F.holiday_counters, offset=forecast_offset)
x_promo = grouped['Promo'].apply(F.days_in_promotion, offset=forecast_offset)
x_promo2 = grouped['Promo2'].apply(F.days_in_promotion, offset=forecast_offset)

x_all = pd.concat([x_lag, x_dif, x_window, x_calendar, x_competition, x_state_hol, x_school_hol, x_promo, x_promo2], axis=1)

x_all.sort_index().head(10)

lag_days_1  lag_days_2  diff_days_1  diff_days_2  \
Store Date                                                           
1     2013-01-01         NaN         NaN          NaN          NaN   
      2013-01-02         0.0         NaN          NaN          NaN   
      2013-01-03      5530.0         0.0       5530.0          NaN   
      2013-01-04      4327.0      5530.0      -1203.0       4327.0   
      2013-01-05      4486.0      4327.0        159.0      -1044.0   
      2013-01-06      4997.0      4486.0        511.0        670.0   
      2013-01-07         0.0      4997.0      -4997.0      -4486.0   
      2013-01-08      7176.0         0.0       7176.0       2179.0   
      2013-01-09      5580.0      7176.0      -1596.0       5580.0   
      2013-01-10      5471.0      5580.0       -109.0      -1705.0   

                  rolling_mean_7D  rolling_mean_14D  rolling_mean_30D  year  \
Store Date                                                                    
1     2013-01-01              NaN               NaN               NaN  2013   
      2013-01-02         0.000000          0.000000          0.000000  2013   
      2013-01-03      2765.000000       2765.000000       2765.000000  2013   
      2013-01-04      3285.666667       3285.666667       3285.666667  2013   
      2013-01-05      3585.750000       3585.750000       3585.750000  2013   
      2013-01-06      3868.000000       3868.000000       3868.000000  2013   
      2013-01-07      3223.333333       3223.333333       3223.333333  2013   
      2013-01-08      3788.000000       3788.000000       3788.000000  2013   
      2013-01-09      4585.142857       4012.000000       4012.000000  2013   
      2013-01-10      4576.714286       4174.111111       4174.111111  2013   

                  quarter  month  ...  is_weekend  is_weekday  is_month_end  \
Store Date                        ...                                         
1     2013-01-01        1      1  ...       False        True         False   
      2013-01-02        1      1  ...        True       False         False   
      2013-01-03        1      1  ...        True       False         False   
      2013-01-04        1      1  ...       False        True         False   
      2013-01-05        1      1  ...       False        True         False   
      2013-01-06        1      1  ...       False        True         False   
      2013-01-07        1      1  ...       False        True         False   
      2013-01-08        1      1  ...       False        True         False   
      2013-01-09        1      1  ...        True       False         False   
      2013-01-10        1      1  ...        True       False         False   

                  CompetitionStartDate  DaysToNextHoliday  \
Store Date                                                  
1     2013-01-01                1593.0               77.0   
      2013-01-02                1594.0               76.0   
      2013-01-03                1595.0               75.0   
      2013-01-04                1596.0               74.0   
      2013-01-05                1597.0               73.0   
      2013-01-06                1598.0               72.0   
      2013-01-07                1599.0               71.0   
      2013-01-08                1600.0               70.0   
      2013-01-09                1601.0               69.0   
      2013-01-10                1602.0               68.0   

                  DaysSinceLastHoliday  DaysToNextHoliday  \
Store Date                                                  
1     2013-01-01                  10.0                0.0   
      2013-01-02                  11.0               72.0   
      2013-01-03                  12.0               71.0   
      2013-01-04                  13.0               70.0   
      2013-01-05                  14.0               69.0   
      2013-01-06                  15.0               68.0   
      2013-01-07                  16.0               67.0 